In [1]:
import os

if not os.path.exists("pw18-ticketing"):
    !git clone https://github.com/simonaoliveto83/pw18-ticketing.git

Cloning into 'pw18-ticketing'...
remote: Enumerating objects: 27, done.
remote: Counting objects: 100% (27/27), done.
remote: Compressing objects: 100% (23/23), done.
remote: Total 27 (delta 1), reused 24 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (27/27), 20.82 KiB | 10.41 MiB/s, done.
Resolving deltas: 100% (1/1), done.


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import pandas as pd

# caricamento dati
# df = pd.read_csv("pw18-ticketing/prototipo3/data/tickets_sintetici.csv")
df = pd.read_csv("/content/drive/MyDrive/prototipo3/data/tickets_sintetici.csv")

df["text"] = df["title"] + " " + df["body"]
df["title"] = df["title"].fillna("")
df["title"] = df["title"].str.lower()
df["body"] = df["body"].fillna("")
df["body"] = df["body"].str.lower()
df["text"] = df["title"] + " " + df["body"]

In [4]:
# encoding delle etichette
from sklearn.preprocessing import LabelEncoder

le_cat = LabelEncoder()
le_prio = LabelEncoder()

df["category_enc"] = le_cat.fit_transform(df["category"])
df["priority_enc"] = le_prio.fit_transform(df["priority"])

In [5]:
# train/test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_cat_train, y_cat_test, y_prio_train, y_prio_test = train_test_split(
    df["text"], df["category_enc"], df["priority_enc"],
    test_size=0.2, random_state=42, stratify=df["category_enc"]
)

In [6]:
# TF-IDF vectorization
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=1000)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

In [7]:
# conversione in tensori PyTorch
import torch

X_train_tensor = torch.tensor(X_train_tfidf.toarray(), dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_tfidf.toarray(), dtype=torch.float32)

y_cat_train_tensor = torch.tensor(y_cat_train.values, dtype=torch.long)
y_cat_test_tensor = torch.tensor(y_cat_test.values, dtype=torch.long)

y_prio_train_tensor = torch.tensor(y_prio_train.values, dtype=torch.long)
y_prio_test_tensor = torch.tensor(y_prio_test.values, dtype=torch.long)

In [8]:
# definizione del modello: multi-task con due teste
#  un solo encoder testuale
#  due teste di classificazione
import torch.nn as nn

class TicketClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()

        self.shared = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2)
        )

        self.cat_head = nn.Linear(hidden_dim, 3)
        self.prio_head = nn.Linear(hidden_dim, 3)

    def forward(self, x):
        x = self.shared(x)
        cat = self.cat_head(x)
        prio = self.prio_head(x)
        return cat, prio

In [9]:
# inizializzazione
input_dim = X_train_tensor.shape[1]
model = TicketClassifier(input_dim, hidden_dim=128)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [10]:
# training loop
n_epochs = 20

for epoch in range(n_epochs):
    model.train()

    optimizer.zero_grad()

    cat_out, prio_out = model(X_train_tensor)

    loss_cat = criterion(cat_out, y_cat_train_tensor)
    loss_prio = criterion(prio_out, y_prio_train_tensor)

    loss = loss_cat + loss_prio

    loss.backward()
    optimizer.step()

    if epoch % 5 == 0:
        print(f"Epoch {epoch}, Loss cat: {loss_cat.item():.4f}, Loss prio: {loss_prio.item():.4f}, Loss : {loss.item():.4f}")

Epoch 0, Loss cat: 1.0998, Loss prio: 1.1012, Loss : 2.2010
Epoch 5, Loss cat: 1.0671, Loss prio: 1.0876, Loss : 2.1548
Epoch 10, Loss cat: 1.0267, Loss prio: 1.0714, Loss : 2.0980
Epoch 15, Loss cat: 0.9760, Loss prio: 1.0525, Loss : 2.0284


In [11]:
# valutazione
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

model.eval()
with torch.no_grad():
    cat_out, prio_out = model(X_test_tensor)

    cat_pred = torch.argmax(cat_out, dim=1)
    prio_pred = torch.argmax(prio_out, dim=1)

# Accuracy
cat_acc = accuracy_score(y_cat_test, cat_pred.numpy())
prio_acc = accuracy_score(y_prio_test, prio_pred.numpy())

# F1
cat_f1 = f1_score(y_cat_test, cat_pred.numpy(), average='macro')
prio_f1 = f1_score(y_prio_test, prio_pred.numpy(), average='macro')

# Classitication
classificazione_cat = classification_report(y_cat_test, cat_pred)
classificazione_prio = classification_report(y_prio_test, prio_pred)

# confusion matrix
cm_cat = confusion_matrix(y_cat_test, cat_pred.numpy())
cm_prio = confusion_matrix(y_prio_test, prio_pred.numpy())
# Categoria
cm_cat_df = pd.DataFrame(cm_cat,
                         index=le_cat.classes_,
                         columns=le_cat.classes_)
# Priorità
cm_prio_df = pd.DataFrame(cm_prio,
                          index=le_prio.classes_,
                          columns=le_prio.classes_)

print(f"Categoria - Accuracy: {cat_acc * 100:.2f} % F1: {cat_f1 :.2f}")
print("\nClassification Report:")
print(classificazione_cat)
print("Confusion Matrix:\n", cm_cat_df)

print(f"\n\nPriorità - Accuracy: {prio_acc * 100:.2f} % F1: {prio_f1 :.2f}")
print("\nClassification Report:")
print(classificazione_prio)
print("\nConfusion Matrix:\n", cm_prio_df)

Categoria - Accuracy: 100.00 % F1: 1.00

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        32
           1       1.00      1.00      1.00        34
           2       1.00      1.00      1.00        34

    accuracy                           1.00       100
   macro avg       1.00      1.00      1.00       100
weighted avg       1.00      1.00      1.00       100

Confusion Matrix:
                  Amministrazione  Commerciale  Tecnico
Amministrazione               32            0        0
Commerciale                    0           34        0
Tecnico                        0            0       34


Priorità - Accuracy: 56.00 % F1: 0.55

Classification Report:
              precision    recall  f1-score   support

           0       0.45      0.91      0.60        32
           1       0.67      0.52      0.58        31
           2       1.00      0.30      0.46        37

    accuracy                      

In [15]:
# funzione di predizione finale
def predict(text):
    model.eval()

    text = text.lower()
    vec = vectorizer.transform([text])
    tensor = torch.tensor(vec.toarray(), dtype=torch.float32)

    with torch.no_grad():
        cat_out, prio_out = model(tensor)

        cat_pred = torch.argmax(cat_out, dim=1).item()
        prio_pred = torch.argmax(prio_out, dim=1).item()

    return le_cat.inverse_transform([cat_pred])[0], \
           le_prio.inverse_transform([prio_pred])[0]

In [16]:
predict("errore accesso piattaforma urgente")

('Amministrazione', 'alta')